# 07b — Feature Engineering V2 Ablation

Validate Plan 2 candidate features with strict train-only lookup generation and apples-to-apples comparison against the selected baseline recipe.

In [7]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'v2_experiment_utils.py').exists():
    NOTEBOOK_DIR = Path('/home/mo-seif/Documents/GP/ml-service/notebooks')
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from v2_experiment_utils import (
    RANDOM_STATE,
    DEFAULT_QUANTILES_3,
    DEFAULT_QUANTILES_5,
    XGB_BASE_PARAMS,
    LGBM_BASE_PARAMS,
    load_processed_dataset,
    build_primary_splits,
    build_feature_spec,
    fit_train_lookups,
    add_plan2_features,
    make_lgbm_frame,
    fit_label_encoders,
    transform_with_label_encoders,
    split_train_val_indices,
    train_lgbm_quantile_models,
    train_xgb_quantile_models,
    predict_lgbm_quantiles,
    predict_xgb_quantiles,
    blend_prediction_maps,
    evaluate_point_and_interval,
    per_make_model_mape_df,
    make_experiment_dir,
    run_kfold_per_make_model_cv,
)

pd.set_option('display.float_format', lambda value: f'{value:,.4f}')
np.set_printoptions(suppress=True)

## 2) Load Baseline Recipe from 07a
Read the best baseline configuration from `07a_dataset_baselines` and set up the experiment output directory.

In [8]:
BASELINE_DIR = Path('/home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07a_dataset_baselines')
if not (BASELINE_DIR / 'baseline_results_summary.csv').exists():
    raise FileNotFoundError('Run 07a_dataset_baselines.ipynb first so baseline_results_summary.csv exists.')
DECISION_TABLE = pd.read_csv(BASELINE_DIR / 'baseline_results_summary.csv').sort_values(['MAPE_pct', 'MAE']).reset_index(drop=True)
SELECTED = DECISION_TABLE.iloc[0].to_dict()
DATASET_TAG = SELECTED['dataset_tag']
TARGET_COL = SELECTED['target_col']
SPLIT_NAME = SELECTED['split_name']
QUANTILES = DEFAULT_QUANTILES_5
ABLATION_FRAMEWORKS = ['XGBoost', 'LightGBM']
RUN_MODE_TAG = f'{DATASET_TAG}__{SPLIT_NAME}__{TARGET_COL}__full_ablation'
OUT_DIR = make_experiment_dir('07b_feature_engineering_v2_ablation')
PARTIAL_DIR = OUT_DIR / f'_partial_runs__{RUN_MODE_TAG}'
PARTIAL_DIR.mkdir(parents=True, exist_ok=True)
SELECTED

{'dataset_tag': '2026-06-03_008',
 'split_name': 'price_stratified',
 'target_col': 'price_egp_log',
 'framework': 'WeightedEnsemble',
 'MAE': 150593.86325308098,
 'RMSE': 605558.621958141,
 'R2': 0.8875250419963173,
 'MAPE_pct': 11.32582524124563,
 'Within_10pct': 61.24000000000001,
 'Within_15pct': 76.75999999999999,
 'Coverage_80_pct': 85.76,
 'Coverage_90_pct': 85.76,
 'Mean_width_pct': 61.54542372219617}

## 3) Load Dataset and Build Splits

Load the full processed dataset and create train / validation / test splits using the recipe selected from 07a.

In [9]:
df_full = load_processed_dataset(DATASET_TAG)
split_map = build_primary_splits(df_full)
train_idx, test_idx = split_map[SPLIT_NAME]
train_idx_fit, val_idx = split_train_val_indices(train_idx, val_fraction=0.10, random_state=RANDOM_STATE)

train_df_base = df_full.loc[train_idx_fit].copy()
val_df_base = df_full.loc[val_idx].copy()
test_df_base = df_full.loc[test_idx].copy()

print('Dataset tag:', DATASET_TAG)
print('Target:', TARGET_COL)
print('Split:', SPLIT_NAME)
print('Train / Val / Test:', len(train_df_base), len(val_df_base), len(test_df_base))

Dataset tag: 2026-06-03_008
Target: price_egp_log
Split: price_stratified
Train / Val / Test: 8999 999 2500


## 4) Plan 2 Ablation Ladder

Define the step-by-step feature additions so we can see which derived features actually help.

In [10]:
ABLATIONS = [
    ('baseline_v1', []),
    ('plus_log_mileage_km', ['log_mileage_km']),
    ('plus_mileage_ratio', ['log_mileage_km', 'mileage_ratio']),
    ('plus_year_bucket', ['log_mileage_km', 'mileage_ratio', 'year_bucket']),
    ('plus_make_model_count', ['log_mileage_km', 'mileage_ratio', 'year_bucket', 'make_model_count']),
    ('full_plan2_bundle', ['log_mileage_km', 'mileage_ratio', 'year_bucket', 'make_model_count', 'mm_price_tier']),
]
ABLATIONS

[('baseline_v1', []),
 ('plus_log_mileage_km', ['log_mileage_km']),
 ('plus_mileage_ratio', ['log_mileage_km', 'mileage_ratio']),
 ('plus_year_bucket', ['log_mileage_km', 'mileage_ratio', 'year_bucket']),
 ('plus_make_model_count',
  ['log_mileage_km', 'mileage_ratio', 'year_bucket', 'make_model_count']),
 ('full_plan2_bundle',
  ['log_mileage_km',
   'mileage_ratio',
   'year_bucket',
   'make_model_count',
   'mm_price_tier'])]

## 5) Ablation Training Function

This helper trains three quantile-regression models for a given feature bundle: **LightGBM**, **XGBoost**, and a **Weighted Ensemble** (55 % XGB / 45 % LGBM). It returns the evaluation metrics table, the full prediction map, and per-make-model MAPE diagnostics.

In [11]:
def run_ablation_step(active_features: list[str]) -> tuple[pd.DataFrame, dict[str, dict[str, np.ndarray]], pd.DataFrame]:
    lookup_df = fit_train_lookups(train_df_base) if {'make_model_count', 'mm_price_tier'} & set(active_features) else None

    train_df = add_plan2_features(train_df_base, lookup_df=lookup_df) if active_features else train_df_base.copy()
    val_df = add_plan2_features(val_df_base, lookup_df=lookup_df) if active_features else val_df_base.copy()
    test_df = add_plan2_features(test_df_base, lookup_df=lookup_df) if active_features else test_df_base.copy()

    num_cols, cat_cols, feature_cols = build_feature_spec(active_features)

    # LightGBM frames
    X_train_lgb = make_lgbm_frame(train_df, feature_cols, cat_cols)
    X_val_lgb = make_lgbm_frame(val_df, feature_cols, cat_cols)
    X_test_lgb = make_lgbm_frame(test_df, feature_cols, cat_cols)

    # XGBoost frames
    label_encoders = fit_label_encoders(train_df, cat_cols)
    X_train_xgb = transform_with_label_encoders(train_df, feature_cols, cat_cols, label_encoders)
    X_val_xgb = transform_with_label_encoders(val_df, feature_cols, cat_cols, label_encoders)
    X_test_xgb = transform_with_label_encoders(test_df, feature_cols, cat_cols, label_encoders)

    lgbm_models = train_lgbm_quantile_models(
        X_train_lgb,
        train_df[TARGET_COL],
        X_val_lgb,
        val_df[TARGET_COL],
        cat_cols,
        QUANTILES,
        params=LGBM_BASE_PARAMS,
        n_estimators_max=1200,
        early_stopping_rounds=40,
    )

    xgb_models = train_xgb_quantile_models(
        X_train_xgb,
        train_df[TARGET_COL],
        X_val_xgb,
        val_df[TARGET_COL],
        QUANTILES,
        params=XGB_BASE_PARAMS,
        n_estimators_max=1200,
        early_stopping_rounds=40,
    )

    lgbm_pred_map = predict_lgbm_quantiles(lgbm_models, X_test_lgb)
    xgb_pred_map = predict_xgb_quantiles(xgb_models, X_test_xgb)
    ensemble_pred_map = blend_prediction_maps([xgb_pred_map, lgbm_pred_map], weights=[0.55, 0.45])

    prediction_maps = {
        'LightGBM': lgbm_pred_map,
        'XGBoost': xgb_pred_map,
        'WeightedEnsemble': ensemble_pred_map,
    }

    metric_rows = []
    is_log_target = TARGET_COL == 'price_egp_log'
    for framework, pred_map in prediction_maps.items():
        metrics = evaluate_point_and_interval(
            test_df[TARGET_COL].to_numpy(),
            pred_map,
            is_log_target=is_log_target,
        )
        metric_rows.append({'framework': framework, **metrics})

    metrics_df = pd.DataFrame(metric_rows).sort_values(['MAPE_pct', 'MAE']).reset_index(drop=True)
    best_framework = metrics_df.iloc[0]['framework']
    median_key = 'median' if 'median' in prediction_maps[best_framework] else 'q50'
    median_pred = np.asarray(prediction_maps[best_framework][median_key], dtype=float)
    if is_log_target:
        if float(np.nanmedian(test_df[TARGET_COL].to_numpy())) < 10:
            median_pred = np.power(10.0, median_pred)
        else:
            median_pred = np.exp(median_pred)
    diag_df = per_make_model_mape_df(test_df, median_pred, min_rows=5)
    return metrics_df, prediction_maps, diag_df

## 6) Run Feature Ablations

Execute the full Plan 2 ablation ladder and compare XGBoost, LightGBM, and the weighted ensemble at each step.

In [12]:
ablation_results: list[pd.DataFrame] = []
ablation_diags: dict[str, pd.DataFrame] = {}
ablation_feature_map: dict[str, str] = {}

for ablation_name, feature_list in ABLATIONS:
    partial_metrics_path = PARTIAL_DIR / f'metrics__{ablation_name}.csv'
    partial_diag_path = PARTIAL_DIR / f'diag__{ablation_name}.csv'

    if partial_metrics_path.exists() and partial_diag_path.exists():
        print(f'Loading cached ablation: {ablation_name}')
        metrics_df = pd.read_csv(partial_metrics_path)
        diag_df = pd.read_csv(partial_diag_path)
    else:
        print(f'Starting ablation: {ablation_name}')
        metrics_df, _, diag_df = run_ablation_step(feature_list)
        metrics_df.to_csv(partial_metrics_path, index=False)
        diag_df.to_csv(partial_diag_path, index=False)
        print(f'Completed ablation: {ablation_name}')

    metrics_df = metrics_df.copy()
    metrics_df.insert(0, 'ablation_name', ablation_name)
    ablation_results.append(metrics_df)
    ablation_diags[ablation_name] = diag_df
    ablation_feature_map[ablation_name] = ','.join(feature_list)

ablation_results_df = pd.concat(ablation_results, ignore_index=True)
display(ablation_results_df.sort_values(['MAPE_pct', 'MAE']).reset_index(drop=True))

Starting ablation: baseline_v1


Completed ablation: baseline_v1
Starting ablation: plus_log_mileage_km
Completed ablation: plus_log_mileage_km
Starting ablation: plus_mileage_ratio
Completed ablation: plus_mileage_ratio
Starting ablation: plus_year_bucket
Completed ablation: plus_year_bucket
Starting ablation: plus_make_model_count
Completed ablation: plus_make_model_count
Starting ablation: full_plan2_bundle
Completed ablation: full_plan2_bundle


,ablation_name,framework,MAE,RMSE,R2,MAPE_pct,Within_10pct,Within_15pct,Coverage_80_pct,Coverage_90_pct,Mean_width_pct
0,plus_make_model_count,WeightedEnsemble,"149,547.6904","598,313.0892",0.8902,11.2461,61.6000,77.0000,72.6400,86.0000,35.8934
1,full_plan2_bundle,WeightedEnsemble,"149,772.7479","594,168.4957",0.8917,11.2574,62.3200,76.9200,71.8800,85.7600,35.2740
2,plus_log_mileage_km,WeightedEnsemble,"149,526.9805","599,086.7220",0.8899,11.2604,61.8400,76.9200,72.4000,86.4000,36.0369
3,baseline_v1,WeightedEnsemble,"150,593.8633","605,558.6220",0.8875,11.3258,61.2400,76.7600,72.4400,86.1200,35.1703
4,plus_year_bucket,WeightedEnsemble,"150,969.2976","602,399.8608",0.8887,11.3342,61.2800,76.8400,73.2000,85.6800,36.1814
5,plus_mileage_ratio,WeightedEnsemble,"150,848.7206","603,086.0626",0.8884,11.3524,61.6400,76.4400,72.1600,85.5200,35.4904
6,plus_log_mileage_km,XGBoost,"151,114.1149","601,332.0973",0.8891,11.4465,61.4400,76.0800,72.8800,87.7600,39.1249
7,full_plan2_bundle,LightGBM,"152,388.7394","604,927.1439",0.8878,11.4581,61.9200,76.5200,67.0800,80.7200,31.5719
8,plus_make_model_count,XGBoost,"152,448.8207","603,658.0135",0.8882,11.4600,61.0400,76.2800,73.2400,88.0800,38.6699
9,full_plan2_bundle,XGBoost,"153,345.7986","596,414.4643",0.8909,11.4653,60.6800,75.9600,72.4800,86.8400,38.0058


## 7) Select Winning Ablation

Pick the best feature bundle by lowest MAPE and show the improvement over the baseline.

In [13]:
winner_table = ablation_results_df.loc[ablation_results_df.groupby('ablation_name')['MAPE_pct'].idxmin()].sort_values('MAPE_pct').reset_index(drop=True)
best_ablation = winner_table.iloc[0].to_dict()
best_ablation['feature_list'] = ablation_feature_map[best_ablation['ablation_name']]
display(winner_table)

baseline_mape = ablation_results_df.loc[ablation_results_df['ablation_name']=='baseline_v1', 'MAPE_pct'].min()
winner_mape = best_ablation['MAPE_pct']
print(f'Improvement over baseline_v1: {baseline_mape - winner_mape:.2f} pp MAPE')

,ablation_name,framework,MAE,RMSE,R2,MAPE_pct,Within_10pct,Within_15pct,Coverage_80_pct,Coverage_90_pct,Mean_width_pct
0,plus_make_model_count,WeightedEnsemble,"149,547.6904","598,313.0892",0.8902,11.2461,61.6000,77.0000,72.6400,86.0000,35.8934
1,full_plan2_bundle,WeightedEnsemble,"149,772.7479","594,168.4957",0.8917,11.2574,62.3200,76.9200,71.8800,85.7600,35.2740
2,plus_log_mileage_km,WeightedEnsemble,"149,526.9805","599,086.7220",0.8899,11.2604,61.8400,76.9200,72.4000,86.4000,36.0369
3,baseline_v1,WeightedEnsemble,"150,593.8633","605,558.6220",0.8875,11.3258,61.2400,76.7600,72.4400,86.1200,35.1703
4,plus_year_bucket,WeightedEnsemble,"150,969.2976","602,399.8608",0.8887,11.3342,61.2800,76.8400,73.2000,85.6800,36.1814
5,plus_mileage_ratio,WeightedEnsemble,"150,848.7206","603,086.0626",0.8884,11.3524,61.6400,76.4400,72.1600,85.5200,35.4904


Improvement over baseline_v1: 0.08 pp MAPE


## 8) Per-Make-Model Diagnostics

Inspect the strongest feature bundle on the full make/model MAPE diagnostic slice.

In [14]:
best_diag_name = best_ablation['ablation_name']
print('Winning ablation:', best_diag_name)

print('\nBest 20 per-make-model (lowest MAPE):')
display(ablation_diags[best_diag_name].sort_values('MAPE_pct', ascending=True).head(20))

print('\nWorst 20 per-make-model (highest MAPE):')
display(ablation_diags[best_diag_name].sort_values('MAPE_pct', ascending=False).head(20))

Winning ablation: plus_make_model_count

Best 20 per-make-model (lowest MAPE):


,make,model,n_test,MAPE_pct
159,Land Rover,Range Rover Velar,5,2.5478
158,Chery,Tiggo 3,11,3.1919
157,MG,ZS,12,3.4168
156,Mercedes,GLC300,14,3.6536
155,Audi,A3,8,3.8235
154,Hyundai,Tucson,25,4.1070
153,Hyundai,Accent RB,7,4.5113
152,MG,6,12,4.7061
151,Skoda,Kodiaq,24,4.8014
150,Land Rover,Range Rover Evoque,8,4.9455



Worst 20 per-make-model (highest MAPE):


,make,model,n_test,MAPE_pct
0,Chevrolet,Pickup,5,42.1874
1,Mazda,323,6,40.9229
2,Peugeot,504,6,38.9866
3,Suzuki,Swift,8,34.7709
4,Honda,Civic,10,29.4796
5,Peugeot,405,9,28.4343
6,Mercedes,200,8,27.2173
7,Fiat,128,11,23.3957
8,Mercedes,S500,5,22.9517
9,Fiat,Siena,11,21.8268


#### Best & Worst by R²

Same winning ablation, sorted by per-make-model R² instead of MAPE.

In [19]:
from sklearn.metrics import r2_score

# Re-run winning ablation to recover predictions for R² diagnostics
win_features = best_ablation['feature_list'].split(',') if best_ablation['feature_list'] else []
_, best_pred_map, _ = run_ablation_step(win_features)

best_fw = best_ablation['framework']
median_key = 'median' if 'median' in best_pred_map[best_fw] else 'q50'
median_pred = np.asarray(best_pred_map[best_fw][median_key], dtype=float)

if TARGET_COL == 'price_egp_log':
    if float(np.nanmedian(test_df_base[TARGET_COL].to_numpy())) < 10:
        median_pred = np.power(10.0, median_pred)
    else:
        median_pred = np.exp(median_pred)

# Build per-make-model R² table
test_df_r2 = test_df_base[['make', 'model', 'price_egp']].copy()
test_df_r2['pred_egp'] = median_pred

r2_rows = []
for (make, model), g in test_df_r2.groupby(['make', 'model'], dropna=False):
    if len(g) < 5:
        continue
    yt = g['price_egp'].to_numpy()
    yp = g['pred_egp'].to_numpy()
    r2_rows.append({
        'make': make,
        'model': model,
        'n_test': len(g),
        'R2': float(r2_score(yt, yp)),
        'MAPE_pct': float(np.mean(np.abs((yt - yp) / np.maximum(np.abs(yt), 1e-9))) * 100,)})

r2_df = pd.DataFrame(r2_rows).sort_values('R2', ascending=False).reset_index(drop=True)

print('Best 20 per-make-model (highest R2):')
display(r2_df.head(20))

print('\nWorst 20 per-make-model (lowest R2):')
display(r2_df.tail(20))

Best 20 per-make-model (highest R2):


,make,model,n_test,R2,MAPE_pct
0,Mercedes,E200,35,0.9817,12.7661
1,Audi,A3,8,0.9782,3.8235
2,Peugeot,408,5,0.9763,8.2763
3,Land Rover,Range Rover Velar,5,0.9752,2.5478
4,BMW,318i,8,0.9738,6.0129
5,Opel,Corsa,15,0.9724,15.5007
6,Volkswagen,Golf,12,0.9721,10.5299
7,Land Rover,Range Rover Evoque,8,0.9720,4.9455
8,Skoda,Octavia,18,0.9677,6.8457
9,Mercedes,C180,51,0.9650,7.5100



Worst 20 per-make-model (lowest R2):


,make,model,n_test,R2,MAPE_pct
140,Mercedes,S450,5,-0.0060,12.7625
141,Mercedes,GLE450,7,-0.0204,7.9831
142,Renault,Megane,28,-0.0357,12.5195
143,Speranza,A516,13,-0.0519,13.3910
144,Fiat,Siena,11,-0.0855,21.8268
145,Xiaomi,Yu7,5,-0.1046,10.4910
146,Jeep,Grand Cherokee,16,-0.1064,13.2813
147,Skoda,Octavia A4,9,-0.1103,12.2029
148,Chery,Envy,11,-0.1599,8.7536
149,Peugeot,504,6,-0.1623,38.9866


## 9) Cross-Validation Setup

Prepare the winning feature bundle and select the correct framework for 5-fold cross-validation.

In [15]:
# Prepare data for cross-validation on the winning ablation
winning_features = best_ablation['feature_list'].split(',') if best_ablation['feature_list'] else []
lookup_df = fit_train_lookups(train_df_base) if {'make_model_count', 'mm_price_tier'} & set(winning_features) else None
cv_df_full = add_plan2_features(df_full, lookup_df=lookup_df)

num_cols, cat_cols, feature_cols = build_feature_spec(winning_features)

# Pick the right framework and params for CV
cv_framework = best_ablation['framework']
cv_params = XGB_BASE_PARAMS
if cv_framework == 'LightGBM':
    cv_params = LGBM_BASE_PARAMS
elif cv_framework == 'WeightedEnsemble':
    cv_framework = 'XGBoost'
    cv_params = XGB_BASE_PARAMS
    print('Note: CV uses XGBoost fallback because ensemble CV is not yet supported.')

print('CV framework:', cv_framework)
print('CV features:', winning_features)

Note: CV uses XGBoost fallback because ensemble CV is not yet supported.
CV framework: XGBoost
CV features: ['log_mileage_km', 'mileage_ratio', 'year_bucket', 'make_model_count']


## 10) Run 5-Fold CV

Execute K-fold cross-validation and collect per-make-model out-of-fold MAPE diagnostics.

In [16]:
cv_diag_df, cv_overall = run_kfold_per_make_model_cv(
    cv_df_full,
    feature_cols,
    cat_cols,
    TARGET_COL,
    framework=cv_framework,
    params=cv_params,
    n_splits=5,
    random_state=RANDOM_STATE,
    min_rows=5,
)

print('\n=== Cross-Validation Overall Metrics ===')
for k, v in cv_overall.items():
    print(f'  {k}: {v:,.4f}')

print('\nCV per-make-model combos:', len(cv_diag_df))

print('\nCV Best 20 (lowest MAPE):')
display(cv_diag_df.sort_values('MAPE_pct', ascending=True).head(20))

print('\nCV Worst 20 (highest MAPE):')
display(cv_diag_df.sort_values('MAPE_pct', ascending=False).head(20))

cv_out_path = OUT_DIR / 'make_model_mape_cv.csv'
cv_diag_df.to_csv(cv_out_path, index=False)
print('\nSaved CV diagnostics to:', cv_out_path)


=== Cross-Validation Overall Metrics ===
  MAE: 155,320.4721
  RMSE: 505,055.0343
  R2: 0.9226
  MAPE_pct: 11.9787
  Within_10pct: 61.4818
  Within_15pct: 76.3242

CV per-make-model combos: 441

CV Best 20 (lowest MAPE):


,make,model,n,MAPE_pct,MAE,R2,mean_price
0,Volkswagen,Tayron,10,1.2302,"35,526.7677",-0.0701,"2,880,000.0000"
1,Jetour,X90,11,2.2871,"34,420.3134",0.7996,"1,501,363.6364"
2,Kaiyi,X3 Pro,5,2.7133,"25,836.1215",-0.5394,"964,000.0000"
3,Geely,GX3 Pro,8,2.8068,"22,539.4954",0.8453,"818,750.0000"
4,Cupra,Formentor,23,3.2437,"64,483.8954",0.7294,"2,133,435.1739"
5,BAIC,BJ30,12,3.3419,"56,499.7134",0.1502,"1,694,575.0000"
6,Jetour,X70 Plus,9,3.7035,"47,679.3966",-0.1584,"1,297,430.5556"
7,Maserati,Grecale,6,3.7749,"192,408.6019",0.9291,"5,741,666.6667"
8,MG,7,10,4.1646,"69,669.7565",0.5168,"1,638,500.0000"
9,Chery,Arrizo 6,11,4.2053,"42,797.6911",0.0416,"1,029,274.5455"



CV Worst 20 (highest MAPE):


,make,model,n,MAPE_pct,MAE,R2,mean_price
440,Toyota,Cressida,5,89.3489,"123,852.9389",-0.3105,"162,000.0000"
439,Daewoo,Juliet,17,77.2264,"73,486.6575",-11.3610,"247,647.0588"
438,Volvo,XC90,8,62.7176,"932,557.6109",-0.2053,"2,821,875.0000"
437,Lada,2017,10,57.0371,"63,361.0193",-0.7882,"124,300.0000"
436,Daewoo,Espero,7,54.1565,"323,116.8959",-0.1279,"401,428.5714"
435,Dodge,Charger,5,52.3173,"2,029,397.7632",0.3316,"4,096,000.0000"
434,Fiat,132,7,49.4389,"40,246.6261",-310.6492,"80,714.2857"
433,Changan,Benni,8,42.7205,"95,224.1572",-2.9289,"217,500.0000"
432,Tesla,Cybertruck,7,42.4315,"3,378,314.6062",-0.3012,"7,511,428.5714"
431,Peugeot,504,9,40.2885,"40,402.6725",-0.9413,"117,777.7778"



Saved CV diagnostics to: /home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07b_feature_engineering_v2_ablation/make_model_mape_cv.csv


## 11) Export Artifacts

Save the ablation winner table, full results table, and best-run per-make-model MAPE diagnostics to CSV.

In [17]:
winner_table.to_csv(OUT_DIR / 'ablation_winner_table.csv', index=False)
ablation_results_df.to_csv(OUT_DIR / 'ablation_results_full.csv', index=False)
ablation_diags[best_diag_name].to_csv(OUT_DIR / f'best_make_model_mape__{best_diag_name}.csv', index=False)
print('Saved ablation artifacts to:', OUT_DIR)

Saved ablation artifacts to: /home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07b_feature_engineering_v2_ablation


## 12) Export Selected Recipe

Save the final selected feature recipe metadata for downstream notebooks.

In [18]:
metadata_df = pd.DataFrame([
    {
        'dataset_tag': DATASET_TAG,
        'target_col': TARGET_COL,
        'split_name': SPLIT_NAME,
        'best_ablation_name': best_ablation['ablation_name'],
        'best_framework': best_ablation['framework'],
        'best_feature_list': best_ablation['feature_list'],
    }
])
metadata_df.to_csv(OUT_DIR / 'selected_ablation_recipe.csv', index=False)
print('Saved selected recipe to:', OUT_DIR / 'selected_ablation_recipe.csv')

Saved selected recipe to: /home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07b_feature_engineering_v2_ablation/selected_ablation_recipe.csv
